In [2]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# Create an API client
from anthropic import Anthropic
client = Anthropic()
model = "claude-sonnet-4-6"

In [4]:
#Helper functions
import json 
def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)

def add_user_message(messages, text):
    user_message = {"role":"user","content":text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role":"assistant","content":text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model" : model,
        "max_tokens" : 1000,
        "messages" : messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }
    if system:
        params["system"] = system
        
    message = client.messages.create(**params)
    return message.content[0].text    

In [5]:
# Tools and Schemas
from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    if unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    if unit == "hours":
        new_date = date + timedelta(hours=duration)
    if unit == "days":
        new_date = date + timedelta(days=duration)
    if unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    if unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    if unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")
        
    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")

def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")
    

In [9]:
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_forma cannot be empty")
    return datetime.now().strftime(date_format)

get_current_datetime("")

ValueError: date_forma cannot be empty